In [1]:
# ============================================================
# RETRIEVE EXISTING FABRIC SLICE
# ============================================================

from fabrictestbed_extensions.fablib.fablib import FablibManager
import json
import traceback

# Initialize Fablib Manager
fablib = FablibManager(
    token_location="id_token.json",
    project_name="Computer_Networks"
)

fablib.show_config()

slice_name = "MySlice-tunnels-static-routes_VISESH"

try:
    print(f"\n[INFO] Attempting to retrieve slice: {slice_name}")
    
    slice = fablib.get_slice(slice_name)
    
    print(f"[SUCCESS] Slice retrieved: {slice_name}")
    print(slice)
    
    print("\n[INFO] Listing slice resources...")
    for node in slice.get_nodes():
        print(f"\nNode: {node.get_name()}")
        for iface in node.get_interfaces():
            print(f"  - Interface: {iface.get_name()}  |  IP: {iface.get_ip_addr()}")

except Exception as e:
    print(f"[ERROR] Could not retrieve slice '{slice_name}'")
    traceback.print_exc()


User: vbent01@jaguar.tamu.edu bastion key is valid!
Configuration is valid


Version,1.9.3
Project Name,Computer_Networks
Token File,id_token.json
Log Level,INFO
Log File,/tmp/fablib/fablib.log
Data directory,/tmp/fablib
Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Bastion Host,bastion.fabric-testbed.net



[INFO] Attempting to retrieve slice: MySlice-tunnels-static-routes_VISESH
[SUCCESS] Slice retrieved: MySlice-tunnels-static-routes_VISESH
-----------  ------------------------------------
Slice Name   MySlice-tunnels-static-routes_VISESH
Slice ID     784b23b8-263b-4973-8aa5-c109a15badda
Slice State  StableOK
Lease End    2025-11-30 17:01:04 +0000
-----------  ------------------------------------

[INFO] Listing slice resources...

Node: Node3
  - Interface: Node3-nic1-p1  |  IP: 192.168.2.2

Node: Node1
  - Interface: Node1-nic1-p1  |  IP: 192.168.1.1

Node: Node2
  - Interface: Node2-nic2-p1  |  IP: 192.168.2.1
  - Interface: Node2-nic1-p1  |  IP: 192.168.1.2


In [2]:
# ------------------------------------------------------------
# 1. Detect nodes by # of networks (1-net, 2-net)
# ------------------------------------------------------------

nodes = slice.get_nodes()

node_left = None     # net1 only
node_right = None    # net2 only
node_router = None   # net1 + net2

for n in nodes:
    ifaces = n.get_interfaces()
    nets = {iface.get_network().get_name() for iface in ifaces}

    if len(nets) == 2:
        node_router = n
    elif len(nets) == 1:
        # Later we classify whether it's net1 or net2
        iface = ifaces[0]
        netname = iface.get_network().get_name()
        ip = str(iface.get_ip_addr())

        if ip.startswith("192.168.1."):
            node_left = n
        elif ip.startswith("192.168.2."):
            node_right = n

print("\n[OK] Nodes identified:")
print("  node1 (left/net1)  =", node_left.get_name())
print("  node2 (router)     =", node_router.get_name())
print("  node3 (right/net2) =", node_right.get_name())

node1 = node_left
node2 = node_router
node3 = node_right

# ------------------------------------------------------------
# 2. Discover the network names (net1, net2)
# ------------------------------------------------------------

router_ifaces = node2.get_interfaces()
router_nets = list({iface.get_network().get_name() for iface in router_ifaces})

# Extract both networks first
netA, netB = router_nets[0], router_nets[1]

# Determine which is net1 and which is net2 by checking IP pattern
ipA = str(node2.get_interface(network_name=netA).get_ip_addr())
ipB = str(node2.get_interface(network_name=netB).get_ip_addr())

if ipA.startswith("192.168.1."):
    net1_name = netA
    net2_name = netB
else:
    net1_name = netB
    net2_name = netA

print("\n[OK] Networks detected:")
print("  net1 =", net1_name)
print("  net2 =", net2_name)

# ------------------------------------------------------------
# 3. Extract IP addresses for all nodes
# ------------------------------------------------------------

node1_net1_addr = str(node1.get_interface(network_name=net1_name).get_ip_addr())
node2_net1_addr = str(node2.get_interface(network_name=net1_name).get_ip_addr())

node2_net2_addr = str(node2.get_interface(network_name=net2_name).get_ip_addr())
node3_net2_addr = str(node3.get_interface(network_name=net2_name).get_ip_addr())

print("\n[OK] IP mapping:")
print("  node1_net1 =", node1_net1_addr)
print("  node2_net1 =", node2_net1_addr)
print("  node2_net2 =", node2_net2_addr)
print("  node3_net2 =", node3_net2_addr)

# ------------------------------------------------------------
# 4. Auto-generate subnets
# ------------------------------------------------------------

net1_subnet = ".".join(node1_net1_addr.split(".")[:3]) + ".0/24"
net2_subnet = ".".join(node3_net2_addr.split(".")[:3]) + ".0/24"

print("\n[OK] Subnets:")
print("  net1_subnet =", net1_subnet)
print("  net2_subnet =", net2_subnet)

print("\n=========== AUTO-DISCOVERY COMPLETE ===========")


[OK] Nodes identified:
  node1 (left/net1)  = Node1
  node2 (router)     = Node2
  node3 (right/net2) = Node3

[OK] Networks detected:
  net1 = net1
  net2 = net2

[OK] IP mapping:
  node1_net1 = 192.168.1.1
  node2_net1 = 192.168.1.2
  node2_net2 = 192.168.2.1
  node3_net2 = 192.168.2.2

[OK] Subnets:
  net1_subnet = 192.168.1.0/24
  net2_subnet = 192.168.2.0/24

=========== AUTO-DISCOVERY COMPLETE ===========


In [3]:
# ============================================================
# M3 FINAL: AUTO-CONTROLLER (sshuttle / WireGuard / Static)
# ============================================================

import time
import re
import random

# ------------------------------------------------------------
# Helper: run commands on nodes
# ------------------------------------------------------------
def run(n, cmd):
    out, err = n.execute(cmd, quiet=True)
    return out.strip()

CURRENT_MODE = None

In [4]:
# ============================================================
# CLEAN WORKING MODE SET (NO EXTRA SPACES)
# ============================================================

CURRENT_MODE = None

# -------------------------
# STATIC ROUTING
# -------------------------
def use_static():
    global CURRENT_MODE
    print("[MODE] Activating Static Routing...")
    node2.execute("echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")
    node1.execute(f"sudo ip route add {net2_subnet} via {node2_net1_addr} >/dev/null 2>&1 || true")
    node3.execute(f"sudo ip route add {net1_subnet} via {node2_net2_addr} >/dev/null 2>&1 || true")
    CURRENT_MODE = "static"
    print("[MODE] Static routing active")
    return CURRENT_MODE

def stop_static():
    print("[CLEAN] Removing static routes")
    node1.execute(f"sudo ip route del {net2_subnet} >/dev/null 2>&1 || true")
    node3.execute(f"sudo ip route del {net1_subnet} >/dev/null 2>&1 || true")

# -------------------------
# WIREGUARD
# -------------------------
def use_wireguard():
    global CURRENT_MODE
    print("[MODE] Activating WireGuard...")
    for n in [node1, node2, node3]:
        n.execute("sudo wg-quick down wg0 >/dev/null 2>&1 || true")
        n.execute("sudo wg-quick up wg0 >/dev/null 2>&1 || true")
    node2.execute("echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")
    CURRENT_MODE = "wireguard"
    print("[MODE] WireGuard active")
    return CURRENT_MODE

def stop_wireguard():
    print("[CLEAN] Stopping WireGuard")
    for n in [node1, node2, node3]:
        n.execute("sudo wg-quick down wg0 >/dev/null 2>&1 || true")

# -------------------------
# SSHUTTLE (REQUIRES STATIC UNDERLAY)
# -------------------------
def use_sshuttle():
    global CURRENT_MODE
    print("[MODE] Activating sshuttle...")
    use_static()
    node2.execute("echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")
    node3.execute(f"sudo ip route add {net1_subnet} via {node2_net2_addr} >/dev/null 2>&1 || true")
    node1.execute("sudo pkill -f sshuttle >/dev/null 2>&1 || true")
    node1.execute("sudo apt-get update -y >/dev/null 2>&1 && sudo apt-get install -y sshuttle >/dev/null 2>&1")
    SSH_USER = 'rocky'
    cmd = (
        "sudo sshuttle --method=nat "
        "--ssh-cmd 'ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null' "
        f"-r {SSH_USER}@{node2_net1_addr} {net2_subnet} --daemon"
    )
    node1.execute(cmd)
    CURRENT_MODE = "sshuttle"
    print("[MODE] sshuttle active (on top of static)")
    return CURRENT_MODE

def stop_sshuttle():
    print("[CLEAN] Stopping sshuttle")
    node1.execute("sudo pkill -f sshuttle >/dev/null 2>&1 || true")

# -------------------------
# CLEAN ALL
# -------------------------
def clean_all_modes():
    stop_sshuttle()
    stop_wireguard()
    stop_static()
    node2.execute("echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")


In [5]:
# ------------------------------------------------------------
# TRAFFIC GENERATOR (iperf3) — CLEAN VERSION
# ------------------------------------------------------------

def start_traffic():
    print("[TRAFFIC] Starting iperf3 traffic Node1 -> Node3")
    for n in [node1, node3]:
        n.execute("sudo apt-get update -y >/dev/null 2>&1 && sudo apt-get install -y iperf3 >/dev/null 2>&1")
    for n in [node1, node3]:
        n.execute("sudo pkill -f iperf3 || true")
    node3.execute("nohup iperf3 -s >/tmp/iperf3_server.log 2>&1 &")
    node1.execute(f"nohup iperf3 -c {node3_net2_addr} -u -b 20M -t 3600 >/tmp/iperf3_client.log 2>&1 &")

def stop_traffic():
    print("[TRAFFIC] Stopping iperf3 traffic")
    for n in [node1, node3]:
        n.execute("sudo pkill -f iperf3 || true")


In [6]:
# ------------------------------------------------------------
# METRICS (clean)
# ------------------------------------------------------------
def get_metrics():
    cmd = f"ping -c 4 {node3_net2_addr}"
    out, err = node1.execute(cmd, quiet=True)
    text = out + err
    m = re.search(r"= [\d\.]+/([\d\.]+)/", text)
    latency = float(m.group(1)) if m else 999.0
    m2 = re.search(r"(\d+)% packet loss", text)
    loss = float(m2.group(1)) if m2 else 100.0
    if "0 received" in text or "100% packet loss" in text:
        print("[METRICS] Ping failed (tunnel broken)")
        return 999.0, 100.0
    print(f"[METRICS] Latency={latency:.3f} ms, Loss={loss:.1f}%")
    return latency, loss

# ------------------------------------------------------------
# FAULT INJECTION (clean)
# ------------------------------------------------------------
def inject_mode_fault(mode):
    print(f"[FAULT] Injecting mode-specific fault for: {mode}")
    if mode == "static":
        stop_static()
    elif mode == "wireguard":
        for n in [node1, node2, node3]:
            n.execute("sudo wg-quick down wg0 >/dev/null 2>&1 || true")
    elif mode == "sshuttle":
        node1.execute("sudo pkill -f sshuttle >/dev/null 2>&1 || true")
    else:
        print("[FAULT] Unknown mode, no fault injected.")


In [7]:
# ------------------------------------------------------------
# DECISION ENGINE (clean)
# ------------------------------------------------------------
def choose_best(current_mode, latency, loss, need_encryption=False, need_icmp=True):
    if loss > 50 or latency > 900:
        print("[DECISION] HARD FAIL: link is down or very bad")
        if current_mode in ["sshuttle", "wireguard"]:
            return "static"
        if current_mode == "static":
            return "wireguard" if need_encryption else "sshuttle"
        return "static"
    if need_encryption:
        return "wireguard" if (latency < 80 and loss < 10) else "static"
    if need_icmp:
        if latency < 50 and loss < 5:
            return "static"
        if latency < 80 and loss < 10:
            return "wireguard"
        return "sshuttle"
    if latency < 100 and loss < 10:
        return "sshuttle"
    return "wireguard"


In [8]:
# ------------------------------------------------------------
# FAULT INJECTION (clean)
# ------------------------------------------------------------
def clear_faults():
    node2.execute("sudo iptables -F >/dev/null 2>&1 || true")
    node2.execute("echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")

def inject_fault_for_mode(mode):
    print(f"[FAULT] Injecting mode-specific fault for: {mode}")
    if mode == "wireguard":
        node2.execute("sudo wg-quick down wg0 >/dev/null 2>&1 || true")
    elif mode == "static":
        node2.execute("echo 0 | sudo tee /proc/sys/net/ipv4/ip_forward >/dev/null")
        node1.execute(f"sudo ip route del {net2_subnet} >/dev/null 2>&1 || true")
        node3.execute(f"sudo ip route del {net1_subnet} >/dev/null 2>&1 || true")
    elif mode == "sshuttle":
        node1.execute("sudo pkill -f sshuttle >/dev/null 2>&1 || true")
    else:
        print("[FAULT] Unknown mode, no fault injected.")


In [9]:
# ------------------------------------------------------------
# AUTO CONTROLLER LOOP (clean + compact)
# ------------------------------------------------------------
def autopilot(interval=20, iterations=10, need_encryption=False, need_icmp=True, demo_fault_every=4):
    global CURRENT_MODE

    print("\n================ STARTING AUTO CONTROLLER ================\n")

    clean_all_modes()

    print("[TRAFFIC] Starting iperf3 traffic Node1 -> Node3")
    node3.execute("sudo pkill -f iperf3 || true")
    node3.execute("nohup iperf3 -s >/dev/null 2>&1 &")
    time.sleep(1)
    node1.execute(f"nohup iperf3 -c {node3_net2_addr} -t 3600 >/dev/null 2>&1 &")

    if need_encryption:
        CURRENT_MODE = use_wireguard()
    elif need_icmp:
        CURRENT_MODE = use_static()
    else:
        CURRENT_MODE = use_sshuttle()

    history = []

    for cycle in range(1, iterations + 1):
        print(f"=========== CYCLE {cycle}/{iterations} ===========")

        if demo_fault_every and cycle % demo_fault_every == 0:
            inject_mode_fault(CURRENT_MODE)
            time.sleep(2)

        latency, loss = get_metrics()

        history.append({
            "cycle": cycle,
            "mode": CURRENT_MODE,
            "latency_ms": latency,
            "loss_pct": loss
        })

        new_mode = choose_best(
            CURRENT_MODE, latency, loss,
            need_encryption=need_encryption,
            need_icmp=need_icmp
        )

        if new_mode != CURRENT_MODE:
            print(f"[DECISION] Switching {CURRENT_MODE} → {new_mode}")
            if new_mode == "static":
                stop_sshuttle()
                stop_wireguard()
                CURRENT_MODE = use_static()
            elif new_mode == "wireguard":
                stop_sshuttle()
                stop_static()
                CURRENT_MODE = use_wireguard()
            elif new_mode == "sshuttle":
                stop_wireguard()
                CURRENT_MODE = use_sshuttle()
        else:
            print(f"[DECISION] Staying on: {CURRENT_MODE}")

        time.sleep(interval)

    print("[TRAFFIC] Stopping iperf3 traffic")
    node1.execute("sudo pkill -f iperf3 || true")
    node3.execute("sudo pkill -f iperf3 || true")

    clean_all_modes()

    print("\n================ AUTO CONTROLLER COMPLETE ================\n")
    return history


In [10]:
def print_history(history, title="RESULTS"):
    """
    Print controller history in a clean, compact, non-spacey format.
    """
    print(f"\n=== {title} (Compact Mode) ===\n")
    for h in history:
        print(
            f"Cycle {h['cycle']:02d} | "
            f"Mode: {h['mode']:<9} | "
            f"Latency: {h['latency_ms']:.3f} ms | "
            f"Loss: {h['loss_pct']:.1f}%"
        )

In [11]:
print("\n=== RUNNING: SECURE SCENARIO (Encryption + ICMP required) ===")

hist_secure = autopilot(
    interval=20,          # check every 20 seconds
    iterations=10,        # 10 control cycles
    need_encryption=True, # WireGuard preferred
    need_icmp=True,       # ICMP performance matters
    demo_fault_every=3    # break current mode every 3 cycles (for demo)
)

print_history(hist_secure, "SECURE SCENARIO")



=== RUNNING: SECURE SCENARIO (Encryption + ICMP required) ===

================ STARTING AUTO CONTROLLER ================

[CLEAN] Stopping sshuttle
[CLEAN] Stopping WireGuard
[CLEAN] Removing static routes
[TRAFFIC] Starting iperf3 traffic Node1 -> Node3
[MODE] Activating WireGuard...
[MODE] WireGuard active
=========== CYCLE 1/10 ===========
[METRICS] Latency=0.495 ms, Loss=0.0%
[DECISION] Staying on: wireguard
=========== CYCLE 2/10 ===========
[METRICS] Latency=0.477 ms, Loss=0.0%
[DECISION] Staying on: wireguard
=========== CYCLE 3/10 ===========
[FAULT] Injecting mode-specific fault for: wireguard
[METRICS] Latency=999.000 ms, Loss=100.0%
[DECISION] HARD FAIL: link is down or very bad
[DECISION] Switching wireguard → static
[CLEAN] Stopping sshuttle
[CLEAN] Stopping WireGuard
[MODE] Activating Static Routing...
[MODE] Static routing active
=========== CYCLE 4/10 ===========
[METRICS] Latency=0.246 ms, Loss=0.0%
[DECISION] Switching static → wireguard
[CLEAN] Stopping sshuttle
[C

In [12]:
print("\n=== RUNNING: ICMP PRIORITY SCENARIO (Static preferred) ===")
hist_icmp = autopilot(
    interval=20,
    iterations=10,
    need_encryption=False,
    need_icmp=True,
    demo_fault_every=3
)
print_history(hist_icmp, "ICMP PRIORITY SCENARIO")


=== RUNNING: ICMP PRIORITY SCENARIO (Static preferred) ===

================ STARTING AUTO CONTROLLER ================

[CLEAN] Stopping sshuttle
[CLEAN] Stopping WireGuard
[CLEAN] Removing static routes
[TRAFFIC] Starting iperf3 traffic Node1 -> Node3
[MODE] Activating Static Routing...
[MODE] Static routing active
=========== CYCLE 1/10 ===========
[METRICS] Latency=0.250 ms, Loss=0.0%
[DECISION] Staying on: static
=========== CYCLE 2/10 ===========
[METRICS] Latency=0.289 ms, Loss=0.0%
[DECISION] Staying on: static
=========== CYCLE 3/10 ===========
[FAULT] Injecting mode-specific fault for: static
[CLEAN] Removing static routes
[METRICS] Latency=999.000 ms, Loss=100.0%
[DECISION] HARD FAIL: link is down or very bad
[DECISION] Switching static → sshuttle
[CLEAN] Stopping WireGuard
[MODE] Activating sshuttle...
[MODE] Activating Static Routing...
[MODE] Static routing active
[MODE] sshuttle active (on top of static)
=========== CYCLE 4/10 ===========
[METRICS] Latency=0.246 ms, Loss

In [13]:
print("\n=== RUNNING: TCP-ONLY SCENARIO (sshuttle preferred) ===")
hist_tcp = autopilot(
    interval=20,
    iterations=10,
    need_encryption=False,
    need_icmp=False,
    demo_fault_every=3
)
print_history(hist_tcp, "TCP-ONLY SCENARIO")


=== RUNNING: TCP-ONLY SCENARIO (sshuttle preferred) ===

================ STARTING AUTO CONTROLLER ================

[CLEAN] Stopping sshuttle
[CLEAN] Stopping WireGuard
[CLEAN] Removing static routes
[TRAFFIC] Starting iperf3 traffic Node1 -> Node3
[MODE] Activating sshuttle...
[MODE] Activating Static Routing...
[MODE] Static routing active
[MODE] sshuttle active (on top of static)
=========== CYCLE 1/10 ===========
[METRICS] Latency=0.230 ms, Loss=0.0%
[DECISION] Staying on: sshuttle
=========== CYCLE 2/10 ===========
[METRICS] Latency=0.214 ms, Loss=0.0%
[DECISION] Staying on: sshuttle
=========== CYCLE 3/10 ===========
[FAULT] Injecting mode-specific fault for: sshuttle
[METRICS] Latency=0.206 ms, Loss=0.0%
[DECISION] Staying on: sshuttle
=========== CYCLE 4/10 ===========
[METRICS] Latency=0.272 ms, Loss=0.0%
[DECISION] Staying on: sshuttle
=========== CYCLE 5/10 ===========
[METRICS] Latency=0.272 ms, Loss=0.0%
[DECISION] Staying on: sshuttle
=========== CYCLE 6/10 ===========
